# F1TENTH Optimal Raceline Velocity Profile Visualizer
This notebook visualizes the optimal global raceline color-coded by velocity ($v_x$ in m/s) with a colorbar, matching high-end telemetry and YouTube racing visualizations.

In [ ]:
import os
import json
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from pathlib import Path

# MAP_NAME = "track4"
MAP_NAME = "track7th"

repo_root = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
map_dir = repo_root / "assets" / "maps" / MAP_NAME
json_path = map_dir / "global_waypoints.json"
yaml_path = map_dir / f"{MAP_NAME}.yaml"

with open(json_path, 'r') as f:
    wp_data = json.load(f)

traj_key = 'global_traj_wpnts_iqp' if 'global_traj_wpnts_iqp' in wp_data else 'global_traj_wpnts_sp'
wpnts = wp_data[traj_key]['wpnts']

x = np.array([w['x_m'] for w in wpnts])
y = np.array([w['y_m'] for w in wpnts])
v = np.array([w['vx_mps'] for w in wpnts])

# Close trajectory loop
x = np.append(x, x[0])
y = np.append(y, y[0])
v = np.append(v, v[0])

points = np.array([x, y]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)

fig, ax = plt.subplots(figsize=(10, 8), dpi=150)

# Overlay track background if available
if yaml_path.exists():
    with open(yaml_path, 'r') as f:
        ydata = yaml.safe_load(f)
    origin = ydata.get('origin', [0.0, 0.0, 0.0])
    res = ydata.get('resolution', 0.05)
    img_path = map_dir / ydata.get('image', f"{MAP_NAME}.png")
    if not img_path.exists():
        img_path = map_dir / f"{MAP_NAME}_waypoint_gen.png"
    if img_path.exists():
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            h, w = img.shape
            extent = [origin[0], origin[0] + w * res, origin[1], origin[1] + h * res]
            ax.imshow(img, cmap='gray', origin='lower', extent=extent, alpha=0.35)

# Colored raceline
lc = LineCollection(segments, cmap='viridis', norm=plt.Normalize(v.min(), v.max()))
lc.set_array(v)
lc.set_linewidth(4.0)
line = ax.add_collection(lc)

# Colorbar
cbar = fig.colorbar(line, ax=ax, fraction=0.035, pad=0.04)
cbar.set_label('Velocity (m/s)', fontsize=12, fontweight='bold', labelpad=10)

ax.autoscale()
ax.set_aspect('equal')
ax.set_title(f"F1TENTH Velocity Profile — {MAP_NAME}", fontsize=14, fontweight='bold')
plt.show()